# 모델 서빙 프레임워크로 모델 서빙하기

## 1. 과정 소개

### 1.1 교과목 기획 배경
AI 모델은 학습이 끝났다고 해서 곧바로 서비스에 적용되는 것이 아닙니다.  
예를 들어, 고양이와 강아지를 구분하는 모델을 만들었다고 해도,  
- 일반 사용자가 Python 코드를 실행할 줄 알아야 하고  
- GPU 환경이 있어야 하고  
- 모델 파일을 직접 다뤄야 한다면  
→ 사실상 서비스로서 활용하기 어렵습니다.  

그래서 **모델 서빙(Model Serving)** 이 필요합니다.  
즉, **사용자는 요청만 보내면 서버가 모델을 실행하고 결과를 반환하는 구조**를 만드는 것.  

이때 고려해야 할 것은 모델 자체만이 아닙니다.  
- 배포 형태(패키징/컨테이너)  
- API 인터페이스(REST/gRPC 등)  
- 동시성 처리와 확장성(스케일 아웃)  
- 모니터링과 장애 복구  

이번 수업은 먼저 **일반적인 배포 관점에서 모델 서빙의 필요와 구성 요소**를 정리한 뒤,  
실습 파트에서 Triton, vLLM 같은 서빙 프레임워크를 예시로 다룹니다.


## 2. 모델 서빙 프레임워크 개요

### 2.1 왜 모델 서빙이 필요한가?
- **연구용 코드 vs 서비스 환경**  
  - 연구 단계: Colab에서 모델을 불러와 `model.predict(image)` 실행  
  - 서비스 단계: 전 세계 사용자가 동시에 API를 호출  
  - → 이 차이를 메워주는 것이 서빙 프레임워크  

### 2.2 서빙 프레임워크의 역할
- 모델을 안정적으로 배포 (재부팅 후에도 실행 가능)  
- 여러 요청을 동시에 처리 (동시성)  
- 오류 발생 시 자동 복구  
- 클라우드 환경에서 확장  

## 3. Nvidia Triton 이해하기

### 3.1 Triton의 개념과 특징

NVIDIA Triton Inference Server는 GPU 환경에서 다양한 딥러닝 모델을
효율적으로 서빙하기 위한 **추론 전용 서버**입니다.

**주요 특징**

- 다양한 프레임워크 지원 (TensorFlow, PyTorch, ONNX 등)  
- GPU 최적화를 통한 **높은 성능** (추론 속도↑, 메모리 사용량↓)  
- **Dynamic Batching 지원**:
  - 개별 요청을 즉시 처리하지 않고, 짧은 시간 동안 요청을 받아 배치 단위로 GPU에 전달
  - GPU 커널 호출 횟수 감소 -> 처리량 증가

### 3.2 Triton 설치 (Docker 기반)

Triton은 Docker 컨테이너 형태로 제공되며,
GPU 드라이버만 올바르게 설치되어 있다면 별도의 복잡한 설치 과정 없이 사용 가능합니다.

공식 이미지 목록:
https://catalog.ngc.nvidia.com/orgs/nvidia/containers/tritonserver

```bash
# 최신 이미지 (권장하지 않음: 버전 변동 위험)
docker pull nvcr.io/nvidia/tritonserver:latest

# 실습 및 수업용 고정 버전
docker pull nvcr.io/nvidia/tritonserver:25.12-py3
```

### 3.3 Triton 서버 실행 (정석 방식)

Triton 서버는 **컨테이너 생성 시점에 모델을 로딩**한다.
모델 로딩에 실패하면 서버는 즉시 종료된다.

따라서 Triton은 일반적인 서비스 컨테이너처럼
`docker start`로 재시작하는 방식에 적합하지 않다.

#### 3.3.1 모델 저장소 디렉터리 구조

Triton은 다음과 같은 **고정된 디렉터리 규칙**을 사용한다.

```text
/models/
└── resnet18/
    ├── config.pbtxt
    └── 1/
        └── model.onnx
```

- `resnet18` : 모델 이름
- `1`         : 모델 버전 (숫자 디렉터리 필수)
- `model.onnx`: 고정 파일명 (반드시 model.onnx)
- `config.pbtxt`: 모델 입출력, 배치, 인스턴스 설정 파일

> config.pbtxt 파일은 수업 자료에서 별도 파일로 제공


#### 3.3.2 Triton 컨테이너 생성 및 실행

```bash
docker run -d \
  --name triton-server \
  --gpus all \
  -p 8000:8000 \
  -p 8001:8001 \
  -p 8002:8002 \
  -v /home/triton/model_repository:/models \
  nvcr.io/nvidia/tritonserver:25.12-py3 \
  tritonserver --model-repository=/models
```

**포트 설명**
- 8000 : HTTP REST API
- 8001 : gRPC API
- 8002 : Metrics (Prometheus)

**중요 주의사항**
- `~`(홈 디렉터리 축약)은 사용하지 말고 **절대 경로**를 사용
- 컨테이너 생성 시 모델 로딩 실패 → 서버 즉시 종료
- 문제 발생 시 `docker start`가 아닌 **컨테이너 재생성**이 정석

#### 3.3.3 Triton 로그 확인

```bash
docker logs triton-server
```

정상 로딩 시 반드시 다음 메시지가 출력된다.

```text
successfully loaded 'resnet18'
| resnet18 | 1 | READY |
```

### 3.4 Triton 상태 확인 방법 (REST API)

```bash
# 서버 준비 상태 확인 
curl -i http://localhost:8000/v2/health/ready
```
  - HTTP/1.1 200 OK 나오면 정상
  - 404면 모델 이름/경로 문제
  - 503이면 서버/모델이 아직 준비 안 됨

```bash
# 특정 모델 준비 상태 확인
curl -i http://localhost:8000/v2/models/resnet18/ready

# 모델 설정 확인
curl http://localhost:8000/v2/models/resnet18/config
```

## 4. Triton 실습 예시

### 4.1 PyTorch → ONNX 변환 (Dynamic Batch)

실무에서는 **배치 크기 고정 모델이 아닌 Dynamic Batch 모델**을 사용하는 것이 일반적입니다.  
4-2.딥러닝 모델 변환과 양자화.ipynb 수업자료에서 만든 모델을 그대로 사용합니다. 

```python
torch.onnx.export(                     # PyTorch 모델을 ONNX 형식으로 변환
    model,                             # 변환 대상 PyTorch 모델
    sample_input,                      # 더미 입력 텐서 (shape: (1, 3, 224, 224))
    "resnet18.onnx",                   # 출력될 ONNX 모델 파일 경로
    input_names=["input"],             # ONNX 입력 텐서 이름 (input: [-1, 3, 224, 224])
    output_names=["output"],           # ONNX 출력 텐서 이름 (output: [-1, 1000])
    opset_version=18,                  # 사용할 ONNX opset 버전
    do_constant_folding=True,          # 상수 연산을 미리 계산하여 그래프 최적화
    dynamic_axes={                     # 동적 배치 크기 설정 
                                       # 배치 크기가 달라져도 하나의 배치로 묶어 처리해야하기 위해 배치 축을 알려줌
        "input": {0: "batch"},         # input의 0번 축을 batch 차원으로 설정 ([-1, 3, 224, 224])
        "output": {0: "batch"}         # output의 0번 축을 batch 차원으로 설정 ([-1, 1000])
    }
)
```

변환 결과 ONNX 입력/출력 형태:
```text
input  : [-1, 3, 224, 224]
output : [-1, 1000]
```

In [5]:
!pip install tritonclient[http]

Found existing installation: tritonclient 2.64.0
Uninstalling tritonclient-2.64.0:
  Successfully uninstalled tritonclient-2.64.0
  Using cached tritonclient-2.64.0-py3-none-manylinux1_x86_64.whl.metadata (3.0 kB)
Using cached tritonclient-2.64.0-py3-none-manylinux1_x86_64.whl (111 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 23.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 36.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [geventhttpclient][geventhttpclient]


#### Triton 추론 요청 방식 설명

Triton Inference Server는 HTTP 기반 API를 제공하므로  
이론적으로는 `curl` 명령으로도 추론 요청을 보낼 수 있습니다.  

다만 이미지 파일을 **FP32 NCHW 텐서로 변환하여 바이너리 형태로 전송해야 하므로**,  
`curl`만으로 추론을 수행하는 것은 실습 및 개발 단계에서 매우 번거롭습니다.  

따라서 실제로는 Triton에서 제공하는 **Python client(`tritonclient`)** 를 사용해  
아래와 같이 추론 요청을 보내는 방식이 일반적입니다.  

이 예제는 **서빙 엔진(Triton)을 애플리케이션 코드에서 어떻게 호출하는지**를  
가장 단순한 형태로 보여줍니다.

In [2]:
import numpy as np
from PIL import Image
import tritonclient.http as httpclient

client = httpclient.InferenceServerClient("localhost:8000")

img = Image.open("image/dog.jpg").convert("RGB")
img = img.resize((224,224))
img = np.array(img).astype(np.float32) / 255.0
img = img.transpose(2,0,1)
img = np.expand_dims(img, axis=0)

inputs = httpclient.InferInput("input", img.shape, "FP32")
inputs.set_data_from_numpy(img)

outputs = httpclient.InferRequestedOutput("output")

result = client.infer(
    model_name="resnet18",
    inputs=[inputs],
    outputs=[outputs]
)

print(result.as_numpy("output").argmax(axis=1))


[259]


## 5. Triton 고급 기능

### 5.1 Dynamic Batching
- 여러 요청을 **짧은 시간 동안 모아서 한 번에 처리**  
- GPU 효율성 극대화  
- 예: 1명씩 처리 vs 10명을 모아서 동시에 처리  

**config.pbtxt 예시 (Dynamic Batching 활성화)**
```text
name: "resnet18"
platform: "onnxruntime_onnx"
max_batch_size: 32
input [
  {
    name: "input"
    data_type: TYPE_FP32
    dims: [3, 224, 224]
  }
]
output [
  {
    name: "output"
    data_type: TYPE_FP32
    dims: [1000]
  }
]

dynamic_batching {
  preferred_batch_size: [4, 8, 16]
  max_queue_delay_microseconds: 2000
}
```

**예상 효과**
- 짧은 지연(예: 2ms) 동안 요청을 모아 배치 처리
- 처리량(throughput) 증가, GPU 활용률 향상

### 5.2 실무 프로젝트 의뢰 사례: 단일 GPU 서버 동시성 최적화

아래는 실제 현업에서 자주 발생하는 모델 서빙 이슈를 정리한 사례입니다.  
단일 요청은 빠른데, 동시 요청이 늘면 지연이 급격히 커지는 전형적인 상황입니다.

<img src="image/프로젝트의뢰.png">

원문 출처: https://discuss.pytorch.kr/t/tx-5090-cosyvoice-llm-concurrency/8977/1

#### 사례 요약
- 환경: RTX 5090 단일 서버, CosyVoice(TTS) 서빙, TensorRT 적용
- 문제: 단일 요청은 2초 이내 처리되지만, 동시 요청 시 Queue가 쌓여 실시간 처리 실패
- 쟁점: "GPU 독점 사용 특성상 하드웨어 추가 없이는 개선 불가" 주장의 타당성 검증 필요
- 목표: 소프트웨어 레벨에서 동시성 확보 및 처리량 개선

#### 이 사례에서 핵심 포인트
- 한 요청씩 바로 처리하면 지연은 짧아도, 요청이 몰릴 때 전체 대기시간이 길어질 수 있습니다.
- 요청을 아주 짧게 모아 한 번에 처리하면 GPU를 더 효율적으로 사용할 수 있습니다.
- API 서버가 동기 방식이면 요청이 줄 서기 쉬우므로, 비동기 처리 구조가 도움이 됩니다.
- 일반 모델/TTS 서빙은 Triton, LLM 서빙은 vLLM처럼 역할을 나누면 운영이 쉬워집니다.

#### 실무 점검 순서
1. 먼저 속도 저하 구간을 분리해서 측정합니다. (모델 추론, 전처리/후처리, 요청 대기)
2. 요청이 몰리는 시간대의 동시 요청 수를 확인합니다.
3. 서버가 여러 요청을 동시에 받도록 구성되어 있는지 확인합니다.
4. 요청을 아주 짧게 모아 한 번에 처리하도록 설정해 봅니다.
5. 변경 전/후를 비교합니다. (평균 응답시간, 느린 구간 응답시간, 초당 처리량)


## 6. vLLM을 활용한 LLM 서빙 실습

### 6.1 vLLM 개요

vLLM은 대규모 언어 모델(LLM)의 **추론 성능 최적화**를 목적으로 개발된
오픈소스 LLM 서빙 엔진입니다.

Transformer 기반 LLM의 특성을 고려하여,
GPU 메모리 사용을 최소화하면서도 높은 처리량을 제공하는 데 초점을 두고 있습니다.

**vLLM의 핵심 특징은 다음과 같습니다.**
- LLM 추론에 특화된 서빙 엔진
- OpenAI API 호환 인터페이스 제공
- 효율적인 KV Cache 관리
- 높은 동시 요청 처리 성능

### 6.2 vLLM이 필요한 이유

기존의 일반적인 LLM 서빙 방식은 다음과 같은 한계를 가집니다.

- 요청 수가 증가할수록 GPU 메모리 사용량 급증
- 각 요청마다 KV Cache를 독립적으로 관리 → 메모리 낭비
- 배치 크기 증가 시 응답 지연(latency) 발생

vLLM은 이러한 문제를 해결하기 위해 **PagedAttention**이라는 메모리 관리 기법을 사용합니다.

**PagedAttention의 개념**
- KV Cache를 연속 메모리가 아닌 페이지 단위로 관리(쪼개사용)
- 서로 다른 요청 간에 GPU 메모리 공간을 유연하게 공유
- 불필요한 메모리 재할당 감소

이를 통해 vLLM은 다음과 같은 장점을 가집니다.
- 동일 GPU 환경에서 더 많은 동시 요청 처리 가능
- 응답 지연 감소
- 대화형 서비스(Chatbot)에 적합한 성능 제공

### 6.3 Triton과 vLLM의 역할 차이

| 구분 | Triton | vLLM |
|----|----|----|
| 주 용도 | 범용 모델 추론 서버 | LLM 전용 추론 서버 |
| 지원 모델 | CNN, ONNX, TensorRT 등 | Transformer 기반 LLM |
| 추론 방식 | 정형 입력/출력 | 텍스트 생성 (Autoregressive) |
| API 형태 | REST / gRPC | OpenAI 호환 API |
| 활용 예 | 이미지 분류, 음성 인식 | 챗봇, RAG, 에이전트 |

> 실무에서는 **일반 모델은 Triton**,  
> **LLM은 vLLM**으로 역할을 분리하여 사용하는 경우가 많습니다.

### 6.4 실습 목표

이번 실습에서는 vLLM을 활용하여
로컬 환경에서 LLM 서버를 구성하고,
OpenAI API와 동일한 방식으로 모델을 호출해봅니다.

또한, 이전에 구현한 RAG 파이프라인에서
LLM 부분을 vLLM 서버 기반 모델로 교체하는 흐름을 경험합니다.

### 6.5 실습 흐름

1. Hugging Face에서 Gemma-3 1B 모델 사용 권한 신청
2. vLLM 서버 실행 및 LLM API 엔드포인트 생성
3. curl 명령어로 LLM API 호출 테스트
4. LangChain에서 OpenAI 대신 vLLM 서버 사용
5. RAG 파이프라인과 연동

### 6.6 Gemma-3 1B 모델 사용 권한 신청

실습에서 사용할 Gemma 계열 모델은 사용 권한 승인이 필요합니다.

아래 링크에서 Hugging Face 계정으로 신청합니다.  
👉 https://huggingface.co/google/gemma-3-1b-it

- 승인까지 약 1~2시간 소요될 수 있습니다.
- 승인 후 Hugging Face 토큰을 발급받아야 합니다.

### 6.7 Hugging Face 토큰 설정

```python
import os
from dotenv import load_dotenv

load_dotenv("env.txt")
hf_token = os.getenv("HF_TOKEN")

if not hf_token:
    raise ValueError("환경 변수 HF_TOKEN이 설정되어 있지 않습니다.")
```

- HF_TOKEN은 Hugging Face 계정에서 발급받은 토큰입니다.
- 토큰은 환경 변수로 관리하는 것이 권장됩니다.

### 6.8 vLLM 서버 실행

터미널에서 아래 명령어를 실행하여 vLLM 서버를 기동합니다.

```bash
vllm serve google/gemma-3-1b-it \
  --host 0.0.0.0 \
  --port 8000 \
  --cpu-offload-gb 8
```

**옵션 설명**
- `google/gemma-3-1b-it`  
  → Hugging Face 모델 이름 (사전 승인 필요)
- `--host 0.0.0.0`  
  → 외부 요청 수신 허용
- `--port 8000`  
  → API 서버 포트
- `--cpu-offload-gb`  
  → GPU 메모리가 부족한 경우 일부 KV Cache를 CPU 메모리로 오프로딩

서버 실행이 성공하면,
OpenAI와 호환되는 API 엔드포인트가 자동으로 생성됩니다.

### 6.9 curl을 이용한 API 호출 테스트

```bash
curl -X POST "http://localhost:8000/v1/chat/completions" \
  -H "Content-Type: application/json" \
  --data '{
    "model": "google/gemma-3-1b-it",
    "messages": [
      {
        "role": "user",
        "content": "한국의 수도는 어디인가요?"
      }
    ]
  }'
```

정상 동작 시 모델의 응답이 JSON 형태로 반환됩니다.

### 6.10 LangChain에서 vLLM 사용 예시

vLLM은 OpenAI API와 호환되므로,
LangChain에서 OpenAI 대신 동일한 인터페이스로 사용할 수 있습니다.

```python
from langchain.chat_models import ChatOpenAI

llm = ChatOpenAI(
    model_name="google/gemma-3-1b-it",
    openai_api_base="http://localhost:8000/v1",
    openai_api_key="dummy_key",
    temperature=0.7,
    max_tokens=512,
)
```

- vLLM은 API Key 검증을 수행하지 않으므로 임의의 값 사용 가능
- 기존 LangChain 코드 변경 없이 서버 주소만 교체하여 사용 가능

### 6.11 정리

- Triton은 범용 추론 서버입니다.
- vLLM은 LLM 추론에 특화된 서버입니다.
- vLLM은 높은 동시 요청 처리와 빠른 응답에 강점을 가집니다.
- RAG, 챗봇, 에이전트 기반 시스템에서는 vLLM이 특히 효과적입니다.

## 7. 클라우드 배포 실습

### 7.1 목표
- 원하는 모델을 Triton or vllm을 사용하여 클라우드에서 각각 배포
- 가능하면 외부에서 배포한 모델을 사용해보기 (보안 및 포트 개방 등 추가 설정 필요)
- 배포 프레임워크를 사용할 때와 안할 때의 성능(추론 속도, 메모리 사용량) 비교  

### 7.2 발표
- 팀별로 결과 공유 
- 어떤 프레임워크가 더 적합했는지 토론